# Train on Kaggle and export for local CPU inference

Start this notebook from your `efficient_detection/` checkout; code and requirements paths are relative to the current directory. Attach a YOLO or COCO dataset and a D-FINE-L checkpoint as Kaggle Datasets. The upstream D-FINE source is fetched separately. Select a GPU accelerator. Enable Internet for the dependency installation cell. Edit the paths below before running.

The default workflow prepares labels, optionally copy-pastes training objects, pre-trains for 2 epochs, fine-tunes for 30 epochs, and exports a verified FP32 ONNX model. The downloaded local bundle needs no PyTorch or GPU.


In [ ]:
from pathlib import Path

# EDIT THESE PATHS to match your attached Kaggle Datasets.
REPO = Path.cwd().resolve()  # Already inside efficient_detection/
DATA_FORMAT = "yolo"  # 'yolo' or 'coco'
IMAGES = Path("/kaggle/input/fisheye/images")
LABELS = Path("/kaggle/input/fisheye/labels")  # YOLO only
ANNOTATIONS = Path("/kaggle/input/fisheye/train.json")  # COCO only
INITIAL_CHECKPOINT = Path("/kaggle/input/dfine-weights/dfine_l_obj2coco_e25.pth")

# Use your existing validation split if available; otherwise choose camera/random splitting.
VAL_IMAGES = None  # e.g. Path('/kaggle/input/fisheye/images/val')
VAL_LABELS = None  # YOLO validation labels
VAL_ANNOTATIONS = None  # COCO validation JSON
CATEGORY_MAP = None  # Optional JSON file mapping source COCO category IDs to 0..4
SPLIT = "camera"  # Use 'random' for filenames without camera<ID>_<scene>_ prefixes.

WORK = REPO.parent / "dfine_workflow"
PREPARED = WORK / "prepared"
RUNS = WORK / "runs/dfine_l_960"
IMAGE_SIZE = 960  # Use 1600 if memory permits; try 640 for faster local inference.
BATCH_SIZE = 2
ACCUMULATE = 1
FINETUNE_EPOCHS = 30
DO_PRETRAIN = True
USE_COPY_PASTE = True

## Install the additional dependencies
Kaggle supplies PyTorch and torchvision; keep their matched versions. This installs the additional training and ONNX packages. Offline notebooks need these packages attached as wheels instead.


In [ ]:
import subprocess
import sys

assert (REPO / "yolo/train_yolo.py").is_file(), "Change to efficient_detection/ first"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO / "requirements/training.txt"),
    ],
    cwd=REPO,
    check=True,
)
import torch
import torchvision

assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle notebook settings"
assert hasattr(torch.amp, "GradScaler"), "This workflow requires PyTorch >= 2.4"
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))


def run(script, *arguments):
    subprocess.run(
        [
            sys.executable,
            "-u",
            str(REPO / "dfine" / script),
            *map(str, arguments),
        ],
        cwd=REPO,
        check=True,
    )


subprocess.run(
    [sys.executable, str(REPO / "dfine/setup_backend.py")],
    cwd=REPO,
    check=True,
)

## Prepare either YOLO or COCO
The script remaps COCO names/IDs and validates boxes. YOLO class IDs must already follow `bus, bike, car, pedestrian, truck` (0–4). Existing validation splits are preserved. Otherwise, camera splitting keeps entire cameras separate. Symlinks avoid copying the attached images.

If you change dataset paths or splitting settings after preparing data, select a new `PREPARED` directory. Existing prepared data is reused on reruns.


In [ ]:
assert DATA_FORMAT in {"yolo", "coco"}
if not (PREPARED / "train.json").exists():
    arguments = [
        "--format",
        DATA_FORMAT,
        "--images",
        IMAGES,
        "--output",
        PREPARED,
        "--split",
        SPLIT,
        "--mode",
        "symlink",
    ]
    if DATA_FORMAT == "yolo":
        arguments += ["--labels", LABELS]
    else:
        arguments += ["--annotations", ANNOTATIONS]
        if CATEGORY_MAP is not None:
            arguments += ["--category-map", CATEGORY_MAP]
    if VAL_IMAGES is not None:
        arguments += ["--val-images", VAL_IMAGES]
        if DATA_FORMAT == "yolo":
            assert VAL_LABELS is not None
            arguments += ["--val-labels", VAL_LABELS]
        else:
            assert VAL_ANNOTATIONS is not None
            arguments += ["--val-annotations", VAL_ANNOTATIONS]
    run("prepare_data.py", *arguments)
else:
    print("Reusing:", PREPARED)

if USE_COPY_PASTE and not (PREPARED / "train_augmented.json").exists():
    run("copy_paste.py", "--dataset", PREPARED)

## Train and select the best checkpoint
Training uses one GPU, EMA, mixed precision, multiscale resizing, and COCO AP selection. Pre-training optionally uses the augmented labels; fine-tuning uses the clean training split. Precomputed pseudo labels can be part of your input dataset. The external VNPT teacher ensemble is not included.

If `last.pth` exists, the cell resumes that stage. Keep `best.pth` alongside it when attaching a previous run. Use a new `RUNS` directory when changing the schedule or model settings. Reduce batch size/resolution if you encounter a GPU memory error.


In [ ]:
assert INITIAL_CHECKPOINT.is_file(), (
    "Attach a D-FINE-L checkpoint and correct INITIAL_CHECKPOINT"
)
common = [
    "--data",
    PREPARED,
    "--image-size",
    IMAGE_SIZE,
    "--batch-size",
    BATCH_SIZE,
    "--accumulate",
    ACCUMULATE,
    "--workers",
    2,
]
checkpoint = INITIAL_CHECKPOINT
if DO_PRETRAIN:
    stage = RUNS / "pretrain"
    arguments = [*common, "--output", stage, "--stage", "pretrain", "--epochs", 2]
    if USE_COPY_PASTE:
        arguments += ["--train-annotations", PREPARED / "train_augmented.json"]
    arguments += (
        ["--resume", stage / "last.pth"]
        if (stage / "last.pth").is_file()
        else ["--checkpoint", checkpoint]
    )
    run("train.py", *arguments)
    checkpoint = stage / "best.pth"

stage = RUNS / "finetune"
arguments = [*common, "--output", stage, "--epochs", FINETUNE_EPOCHS]
arguments += (
    ["--resume", stage / "last.pth"]
    if (stage / "last.pth").is_file()
    else ["--checkpoint", checkpoint]
)
run("train.py", *arguments)
BEST = stage / "best.pth"
print("Best fine-tuned checkpoint:", BEST)

## Export and verify ONNX
The exporter includes postprocessing, uses EMA weights, checks the ONNX graph, and compares CPU ONNX Runtime results with PyTorch. It also tests one real validation image. Deployment uses a fixed resolution and batch size 1.


In [ ]:
import json

validation = json.loads((PREPARED / "val.json").read_text())
sample = PREPARED / "images" / validation["images"][0]["file_name"]
ONNX = WORK / "fisheye.onnx"
run("export_onnx.py", "--checkpoint", BEST, "--output", ONNX, "--test-image", sample)
run(
    "infer.py",
    "--model",
    ONNX,
    "--input",
    sample,
    "--output",
    WORK / "onnx_smoke_test",
)

## Package for your local machine
Download `fisheye_local.zip` from the Kaggle notebook outputs. Extract it, follow `DFINE_LOCAL.md`, and run `python -m efficient_detection.dfine.infer --model fisheye.onnx --input YOUR_IMAGES --output results`. CPU speed depends on your local machine and model resolution.


In [ ]:
import shutil

bundle = WORK / "fisheye_local"
bundle.mkdir(parents=True, exist_ok=True)
for relative in [
    "__init__.py",
    "dfine/__init__.py",
    "dfine/infer.py",
    "shared/__init__.py",
    "shared/common.py",
]:
    target = bundle / "efficient_detection" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / relative, target)
(bundle / "requirements").mkdir(exist_ok=True)
shutil.copy2(REPO / "requirements/inference.txt", bundle / "requirements/inference.txt")
shutil.copy2(REPO / "docs/DFINE_LOCAL.md", bundle / "DFINE_LOCAL.md")
shutil.copy2(ONNX, bundle / ONNX.name)
shutil.copy2(ONNX.with_suffix(".json"), bundle / ONNX.with_suffix(".json").name)
archive = shutil.make_archive(str(WORK.parent / "fisheye_local"), "zip", bundle)
print("Download:", archive)